# Stochastic Calculus, Step by Step · Lesson 25: Girsanov's theorem

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Reweighting, on a coin first · 3. The density process Z · 4. A new probability from the weights · 5. Girsanov's theorem · 6. Volatility can't be changed · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 25,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. Reweighting, on a coin first

Part 2 changed $p$ to $\tilde p$ by giving each path a weight $Z = \tilde P(\text{path})/P(\text{path})$ (Lesson 12). The same two-step tree shows what's coming: the weights average 1, and any expectation under the new probability is a weighted expectation under the old.

In [ ]:
p_, q_ = 0.6, 0.4
tree_rows = []
for path, ups in (("HH", 2), ("HT", 1), ("TH", 1), ("TT", 0)):
    downs = 2 - ups
    P_ = p_**ups * (1 - p_) ** downs; Q_ = q_**ups * (1 - q_) ** downs
    tree_rows.append({"path": path, "ups": ups, "P": P_, "Q": Q_, "Z": Q_ / P_})
out["coin"] = {"p": p_, "q": q_, "rows": tree_rows, "EZ": sum(x["P"] * x["Z"] for x in tree_rows),
               "E_Q_ups": sum(x["Q"] * x["ups"] for x in tree_rows), "E_PZ_ups": sum(x["P"] * x["Z"] * x["ups"] for x in tree_rows)}
pd.DataFrame(tree_rows)

## 3. The density process Z

In continuous time, the weight of a path up to $t$ is

$$Z_t = \exp\Big(-\theta W_t - \tfrac12\theta^2t\Big), \qquad dZ_t = -\theta Z_t\,dW_t .$$

It is the exponential martingale of Lesson 18, starting at 1, always positive, with average 1 at every $t$. With $\theta = 0.25$ (the share's market price of risk):

In [ ]:
rng = np.random.default_rng(seed)
theta = theta_mpr
paths, n_t = 200_000, 50
dt = T / n_t
dW = rng.normal(0.0, math.sqrt(dt), (paths, n_t))
W = dW.cumsum(axis=1)
tg = dt * np.arange(1, n_t + 1)
Z_all = np.exp(-theta * W - 0.5 * theta**2 * tg)
W_T = W[:, -1]; Z_T = Z_all[:, -1]
out["z"] = {"theta": theta, "means": [{"t": float(tg[i]), "mean": float(Z_all[:, i].mean())} for i in (9, 24, 49)],
            "exact_var": math.exp(theta**2 * T) - 1, "var": float(Z_T.var()), "min": float(Z_T.min()), "max": float(Z_T.max()),
            "at_W_plus1": math.exp(-theta - 0.5 * theta**2), "at_W_minus1": math.exp(theta - 0.5 * theta**2), "at_0": math.exp(-0.5 * theta**2),
            "paths": paths}
out["z"]

## 4. A new probability from the weights

Define $\tilde P(A) = E[Z_T\mathbf 1_A]$. The weights are positive, so events keep the same possibility: the two probabilities are **equivalent**. Under $\tilde P$, paths that end low have more weight, so the distribution of $W_T$ moves down. In fact $Z_T\varphi_T(w)$, the old density times the weight, is exactly a normal density centred at $-\theta T$.

In [ ]:
wg = np.linspace(-3.5, 3.5, 281)
old = phi(wg)
new_exact = old * np.exp(-theta * wg - 0.5 * theta**2)
new_named = phi(wg + theta)
weights = Z_T / Z_T.sum()
ev_rows = []
for name, mask, exact_P, exact_Q in (("W_T > 0", W_T > 0, 0.5, 1 - N(theta)), ("W_T > 1", W_T > 1, 1 - N(1.0), 1 - N(1 + theta)),
                                     ("W_T < -1", W_T < -1, N(-1.0), N(-1 + theta))):
    ev_rows.append({"event": name, "P_sim": float(mask.mean()), "P_exact": exact_P, "Q_sim": float((Z_T * mask).mean()), "Q_exact": exact_Q})
out["newp"] = {"w": wg.tolist(), "old": old.tolist(), "new": new_exact.tolist(), "gap": float(np.abs(new_exact - new_named).max()), "rows": ev_rows,
               "mean_Q": float((Z_T * W_T).mean()), "mean_Q_exact": -theta * T, "var_Q": float((Z_T * W_T**2).mean() - (Z_T * W_T).mean() ** 2)}
pd.DataFrame(ev_rows)

## 5. Girsanov's theorem

Under $\tilde P$, the process $\tilde W_t = W_t + \theta t$ is a standard Brownian motion. Checked by its mean, variance, independence of increments, and martingale property, all with the weights $Z_T$ on the simulated $P$ paths:

In [ ]:
Wt = W + theta * tg
i_h, i_e = 24, 49
inc1 = Wt[:, i_h]; inc2 = Wt[:, i_e] - Wt[:, i_h]
wE = lambda x: float((Z_T * x).mean())
out["girs"] = {"mean_half": wE(inc1), "var_half": wE(inc1**2) - wE(inc1) ** 2, "mean_end": wE(Wt[:, i_e]), "var_end": wE(Wt[:, i_e] ** 2) - wE(Wt[:, i_e]) ** 2,
               "cov_inc": wE(inc1 * inc2) - wE(inc1) * wE(inc2), "mean_plain": float(Wt[:, i_e].mean()),
               "ess": float(Z_T.sum() ** 2 / (Z_T**2).sum() / paths), "ess_theory": math.exp(-theta**2 * T),
               "mgf": wE(np.exp(0.7 * Wt[:, i_e])), "mgf_exact": math.exp(0.5 * 0.7**2)}
# the share's drift under the two probabilities
S_T = S0 * np.exp((mu - 0.5 * sigma**2) * T + sigma * W_T)
out["share"] = {"mean_P": float(S_T.mean()), "mean_P_exact": S0 * math.exp(mu * T), "mean_Q": wE(S_T), "mean_Q_exact": S0 * math.exp(r * T),
                "p_above_P": float((S_T > K).mean()), "p_above_Q": wE(S_T > K), "N_minus": N(float(d_minus(S0, T))),
                "p_above_P_exact": N((math.log(S0 / K) + (mu - 0.5 * sigma**2) * T) / sigma),
                "call_Z": math.exp(-r * T) * wE(np.maximum(S_T - K, 0)), "call_bs": float(call(S0, T))}
out["share_rows"] = [{"what": "Average share price at T", "under_p": out["share"]["mean_P"], "under_q": out["share"]["mean_Q"], "exact_q": out["share"]["mean_Q_exact"]},
                     {"what": "Probability that it ends above K", "under_p": out["share"]["p_above_P"], "under_q": out["share"]["p_above_Q"], "exact_q": out["share"]["N_minus"]}]
{**out["girs"], **out["share"]}

## 6. Volatility can't be changed

Changing probability reweights paths; it doesn't change the paths. The sum of squared moves along a path, its quadratic variation (Lesson 14), is the same number under both probabilities. So $\tilde P$ can change the drift but never the volatility.

In [ ]:
gen = np.random.default_rng(seed + 1)
n_fine, p_small = 2000, 4000
dS_fine = gen.normal(0.0, math.sqrt(T / n_fine), (p_small, n_fine))
qv = (dS_fine**2).sum(axis=1)
W_fine = dS_fine.sum(axis=1)
Zf = np.exp(-theta * W_fine - 0.5 * theta**2 * T)
qv_q = dS_fine + theta * T / n_fine
qv_shift = (qv_q**2).sum(axis=1)
out["qv_paths"], out["qv_steps"] = p_small, n_fine
out["qv"] = {"mean_P": float(qv.mean()), "sd_P": float(qv.std()), "mean_Q": float((Zf * qv).sum() / Zf.sum()), "theory_sd": math.sqrt(2 * T**2 / n_fine),
             "corr_with_Z": float(np.corrcoef(qv, Zf)[0, 1]), "shift_gap": float(np.abs(qv_shift - qv).mean())}
out["qv"]

In [ ]:
# Chart data for the video (no random numbers)
out["newp"]["ticks"] = [{"x": float(x_), "label": f"{x_:.0f}"} for x_ in (-3, -2, -1, 0, 1, 2, 3)]

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 4: Solutions*.

**Exercise 1.** Show directly that $E[Z_T] = 1$ for $Z_T = e^{-\theta W_T - \frac12\theta^2T}$, using the moment generating function of the normal. Then show that $Z_t$ is a martingale by splitting $W_T = W_t + (W_T - W_t)$.

**Exercise 2.** Use Itô's lemma to show $dZ_t = -\theta Z_t\,dW_t$. Why does having no $dt$ term make it a martingale? Find $Z_T$ at $W_T = 1$ and $W_T = -1$ with $\theta = 0.25$.

**Exercise 3.** Check that $\tilde P(A) = E[Z_T\mathbf 1_A]$ is a probability: non-negative, total 1, additive on disjoint events. Why is it equivalent to $P$? What would break if $Z_T$ could be 0?

**Exercise 4.** Compute $\tilde E[e^{u\tilde W_T}]$ as $E[Z_Te^{u(W_T + \theta T)}]$ by combining exponents. Conclude that $\tilde W_T \sim N(0, T)$ under $\tilde P$.

**Exercise 5.** Rewrite $dS = \mu S\,dt + \sigma S\,dW$ in terms of $d\tilde W$ with $\theta = (\mu - r)/\sigma$. Find $S_T$ in terms of $\tilde W_T$, and compute $\tilde P(S_T > K)$ and $P(S_T > K)$ with the course's numbers.

**Exercise 6.** Compute $E[Z_T^2] = e^{\theta^2T}$. Importance-sampling theory says that weights with this second moment are worth about $e^{-\theta^2T}$ of their paths in effective sample size. Evaluate this for $\theta = 0.25$ and for $\theta = 3$. Why is a large change of probability dangerous for a simulation?

**Exercise 7.** (Code) Price the call and the put by simulating under $P$ with $\mu = 8\%$ and weighting each path by $Z_T$; compare with simulating directly under $\tilde P$ and with the formula. Repeat with $\mu = 50\%$. Which standard errors shrink, which grow, and why?

**Exercise 8.** (Code) On 2000 paths of 2000 steps, show that the realised $\sum(\Delta W)^2$ is unchanged by the change of probability, in mean and distribution, by computing its weighted mean and weighted quantiles. Then simulate $\tilde W$ and check that $\tilde W^2_t - t$ has weighted mean 0 at $t = 0.5$ and $t = 1$.

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")